# 00 — Sample Prompt Generation

Inspect the selected sample, persist State A if needed, and build or reuse the State B revision artifacts.

In [ ]:
from pathlib import Path
import json
import logging
import os
import shutil
import sys

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "code_base").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
MODULE_ROOT = PROJECT_ROOT / "code_base" / "fea_cad_one_sample"
if str(MODULE_ROOT) not in sys.path:
    sys.path.insert(0, str(MODULE_ROOT))

import src.interfaces as api

logging.basicConfig(level=logging.INFO, format="%(name)s | %(levelname)s | %(message)s")

sample_id = "00689964"
selection_source = "dataset"
config_name = "config_gpt_5_4_mini.yaml"
connection_string = os.environ.get("CAD_DB_CONNECTION_STRING")
OUTPUT_ROOT = MODULE_ROOT / "outputs"
STATE_A_DIR = OUTPUT_ROOT / f"sample_{sample_id}" / "01_dataset_original"
STATE_B_DIR = OUTPUT_ROOT / f"sample_{sample_id}" / "02_fea_constrained_revision"
STATE_A_VIEWS_DIR = STATE_A_DIR / "views"
STATE_B_VIEWS_DIR = STATE_B_DIR / "views"

print("[SETUP] complete")
print(f"  → module root : {MODULE_ROOT}")
print(f"  → sample_id   : {sample_id}")
print(f"  → state A dir : {STATE_A_DIR}")
print(f"  → state B dir : {STATE_B_DIR}")
print(f"  → selection   : {selection_source}")

In [ ]:
print("[STAGE] sample load")
sample = api.load_selected_sample(
    module_root=MODULE_ROOT,
    sample_id=sample_id,
    selection_source=selection_source,
    connection_string=connection_string,
)
print(f"  ✓ sample_id : {sample.sample_id}")
print(f"  ✓ source    : {sample.source}")
print(f"  ✓ prompt    : {len(sample.prompt.splitlines())} lines")
print(f"  ✓ code      : {len((sample.ground_truth_code or '').splitlines())} lines")
assert sample.sample_id == sample_id
assert isinstance(sample.prompt, str) and sample.prompt.strip()
assert isinstance(sample.ground_truth_code, str) and sample.ground_truth_code.strip()

In [ ]:
print("[STAGE] State A persistence and views")
state_a_prompt_path = STATE_A_DIR / "original_prompt.txt"
state_a_code_path = STATE_A_DIR / "database_original_code.py"
state_a_step_path = STATE_A_DIR / "original.step"
state_a_stl_path = STATE_A_DIR / "original.stl"
state_a_summary_path = STATE_A_DIR / "metadata.json"
state_a_provenance_path = STATE_A_DIR / "provenance.json"
config = api.PipelineConfig(
    config_name=config_name,
    config_path=MODULE_ROOT / "src" / "copied_from_cadcodeverify" / "configs" / config_name,
    output_root=OUTPUT_ROOT,
    force=False,
)

if not all(path.exists() for path in (state_a_prompt_path, state_a_code_path, state_a_step_path, state_a_stl_path)):
    original_code = api.generate_original_code(sample, config)
    original_exec = api.execute_and_export_cadquery(original_code, STATE_A_DIR, basename="original", force=False)
else:
    original_code = state_a_code_path.read_text(encoding="utf-8")
    original_exec = {
        "step_path": str(state_a_step_path),
        "stl_path": str(state_a_stl_path),
        "execution_log_path": str(STATE_A_DIR / "execution_log.txt"),
    }

if not all((STATE_A_VIEWS_DIR / f"{name}.png").exists() for name in ("front", "side", "top", "iso", "grid")):
    original_views = api.render_views(Path(original_exec["stl_path"]), STATE_A_VIEWS_DIR, force=False)
else:
    original_views = {name: str(STATE_A_VIEWS_DIR / f"{name}.png") for name in ("front", "side", "top", "iso", "grid")}

print(f"  ✓ prompt path : {state_a_prompt_path}")
print(f"  ✓ code path   : {state_a_code_path}")
print(f"  ✓ step path   : {original_exec['step_path']}")
print(f"  ✓ stl path    : {original_exec['stl_path']}")
print(f"  ✓ views       : {sorted(original_views.values())}")
print(f"  ✓ summary     : {state_a_summary_path}")
print(f"  ✓ provenance  : {state_a_provenance_path}")
assert state_a_prompt_path.exists()
assert state_a_code_path.exists()
assert Path(original_exec["step_path"]).exists()
assert Path(original_exec["stl_path"]).exists()

In [ ]:
print("[STAGE] State B prompt and revision")
load_case_path = STATE_B_DIR / "load_case.json"
selector_hints_path = STATE_B_DIR / "selector_hints.json"
revision_prompt_path = STATE_B_DIR / "fea_revision_prompt.txt"
revision_code_path = STATE_B_DIR / "fea_revision_code.py"
revision_step_path = STATE_B_DIR / "fea_revision.step"
revision_stl_path = STATE_B_DIR / "fea_revision.stl"
revision_change_log_path = STATE_B_DIR / "fea_revision_change_log.json"
revision_provenance_path = STATE_B_DIR / "provenance.json"

load_case = api.LoadCase(**json.loads(load_case_path.read_text(encoding="utf-8"))) if load_case_path.exists() else api.write_load_case(sample_id, load_case_path, force=False)
selector_hints = api.SelectorHints(**json.loads(selector_hints_path.read_text(encoding="utf-8"))) if selector_hints_path.exists() else api.write_selector_hints(load_case, selector_hints_path, force=False)
built_revision_prompt = api.build_fea_prompt(sample.prompt, original_code, load_case, selector_hints)

if not all(path.exists() for path in (revision_prompt_path, revision_code_path, revision_step_path, revision_stl_path)):
    revision = api.revise_code_for_fea(sample.prompt, original_code, load_case, selector_hints, config)
    revision_code = revision.code_path.read_text(encoding="utf-8")
    revision_exec = api.execute_and_export_fea_revision_cadquery(revision_code, STATE_B_DIR, force=False)
else:
    revision_code = revision_code_path.read_text(encoding="utf-8")
    revision = api.FEARevisionResult(
        sample_id=sample_id,
        prompt_path=revision_prompt_path,
        load_case_path=load_case_path,
        selector_hints_path=selector_hints_path,
        code_path=revision_code_path,
        change_log_path=revision_change_log_path,
        provenance_path=revision_provenance_path,
        step_path=revision_step_path,
        stl_path=revision_stl_path,
        execution_log_path=STATE_B_DIR / "execution_log.txt",
        view_paths={
            "front": STATE_B_VIEWS_DIR / "front.png",
            "side": STATE_B_VIEWS_DIR / "side.png",
            "top": STATE_B_VIEWS_DIR / "top.png",
            "iso": STATE_B_VIEWS_DIR / "iso.png",
            "grid": STATE_B_VIEWS_DIR / "grid.png",
            "annotated_support_load": STATE_B_VIEWS_DIR / "annotated_support_load.png",
        },
        code_hash_sha256="existing",
    )
    revision_exec = {
        "step_path": str(revision_step_path),
        "stl_path": str(revision_stl_path),
        "execution_log_path": str(STATE_B_DIR / "execution_log.txt"),
    }

if not all((STATE_B_VIEWS_DIR / f"{name}.png").exists() for name in ("front", "side", "top", "iso", "grid")):
    revision_views = api.render_views(Path(revision_exec["stl_path"]), STATE_B_VIEWS_DIR, force=False)
else:
    revision_views = {name: str(STATE_B_VIEWS_DIR / f"{name}.png") for name in ("front", "side", "top", "iso", "grid")}
annotated_path = STATE_B_VIEWS_DIR / "annotated_support_load.png"
if not annotated_path.exists():
    annotated_path = api.render_support_load_annotation(Path(revision_exec["stl_path"]), annotated_path, selector_hints, force=False)

print(f"  ✓ load case   : {load_case_path}")
print(f"  ✓ selector    : {selector_hints_path}")
print(f"  ✓ prompt path : {revision.prompt_path}")
print(f"  ✓ code path   : {revision.code_path}")
print(f"  ✓ step path   : {revision_exec['step_path']}")
print(f"  ✓ stl path    : {revision_exec['stl_path']}")
print(f"  ✓ views       : {sorted(revision_views.values())}")
print(f"  ✓ annotation  : {annotated_path}")
print(f"  ✓ built prompt: {len(built_revision_prompt.splitlines())} lines")
assert load_case.sample_id == sample_id
assert selector_hints.sample_id == sample_id
assert revision.prompt_path.exists() or revision_prompt_path.exists()
assert revision.code_path.exists() or revision_code_path.exists()

In [ ]:
print("[STAGE] failure case")
try:
    api.load_selected_sample(module_root=MODULE_ROOT, sample_id=sample_id, selection_source="bogus")
except Exception as exc:
    print(f"  ✓ error type : {type(exc).__name__}")
    print(f"  ✓ message    : {exc}")

## Summary

- State A and State B artifacts are visible through public interfaces.
- The notebook reuses existing outputs when present and only regenerates missing artifacts.
- The failure case shows the selection-source validation path explicitly.